# Avance 3 · Baseline de regresión de la potencia (fase 2)

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este cuaderno es el compañero del [03_Baseline](03_Baseline.ipynb) para la fase 2. El baseline del Avance 3 clasificaba el par compuesto-diana en activo o inactivo a 10 µM; después la variable objetivo cambió: lo que se predice es la **potencia** (`pActividad`) y la clase se deriva de ella al final ([2.9](../docs/decisiones.md#d2-9)). Así que el baseline también es una regresión, y esto es lo que lo establece.

Las preguntas de la rúbrica del Avance 3, aplicadas aquí:

- **¿Qué algoritmo?** Cuatro, que se distinguen por lo que ven: el promedio (el piso), una recta con las mediciones exactas, la misma recta con las censuradas **tomando la cota como si fuera el valor** y la misma recta con las censuradas **como cotas, con su verosimilitud** (regresión de Tobit).
- **¿Qué características pesan?** Los coeficientes de la recta, con y sin censura, y si coinciden.
- **¿Sobreajusta?** El error en entrenamiento contra el de validación.
- **¿Qué métrica?** La concordancia de Harrell, que usa también las censuradas; Spearman y RMSE solo con las exactas, declarado así.
- **¿Qué desempeño mínimo?** No hay un histórico. El piso es el promedio; el techo, el ruido propio de la medición.


## 1. Preparación

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo: la columna de tiempos es comparable

import platform
import sys
import time
from collections.abc import Sequence
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.optimize import minimize
from scipy.special import log_ndtr
from scipy.stats import spearmanr
from sklearn.base import BaseEstimator, clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))                 # solo para el registro de la corrida, al final
PROCESADOS = RAIZ / "data" / "processed"

# Constantes del baseline.
ALFA_RIDGE = 100.0        # penalización de Ridge; el Tobit usa la misma en su escala
CORTE_10_UM = 5.0         # pActividad 5 = 10 µM
MIN_POR_CLASE = 20        # un estrato se reporta si tiene 20 casos de cada clase derivada
# Reproducibilidad experimental de la potencia pública (unidades logarítmicas): el piso del error
# que tiene sentido esperar (`kalliokoski2013ic50`, `kramer2012ki`; `landrum2024ruido` la matiza).
RUIDO_EXPERIMENTAL_IC50 = 0.7
RUIDO_EXPERIMENTAL_KI = 0.5

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

sistema  Linux-6.6.114.1-microsoft-standard-WSL2-x86_64-with-glibc2.43
python   3.12.13
numpy    2.5.4
pandas   3.0.6
sklearn  1.9.1
scipy    1.18.1


### El código

Todo el código que usa este cuaderno está en las celdas de abajo, para que se lea de principio a fin sin abrir otro archivo. **La misma lógica vive en `src/`** (`src/evaluation/regresion.py`, `src/models/tobit.py` y `src/models/baseline_regresion.py`), donde se prueba y de donde la reutilizará el Avance 5; `tests/test_03b_codigo.py` comprueba que las funciones de estas celdas y las de `src/` sean idénticas, de modo que no se separen. Qué hace cada baseline y por qué son cuatro está en la sección 2; aquí está cómo está escrito.

#### Las métricas

La **concordancia de Harrell** compara solo los pares de mediciones cuyos intervalos no se solapan (`superior_i < inferior_j`), porque solo de esos se sabe cuál es más potente; un par es concordante si el modelo predice más potencia para el más potente, y un empate cuenta medio. Es la única métrica que usa también las censuradas. **Spearman y RMSE** se calculan solo con las exactas y devuelven cuántas son, porque una cifra sin su `n` no se puede leer.

In [2]:
def _exactas(limite_inferior: np.ndarray, limite_superior: np.ndarray) -> np.ndarray:
    inf = np.asarray(limite_inferior, dtype=float)
    sup = np.asarray(limite_superior, dtype=float)
    return np.isfinite(inf) & (inf == sup)


def concordancia_harrell(limite_inferior: np.ndarray, limite_superior: np.ndarray,
                         prediccion: np.ndarray, bloque: int = 2000) -> float:
    """Fracción de pares comparables que el modelo ordena bien.

    Un par (i, j) es **comparable** si los intervalos no se solapan
    (`superior_i < inferior_j`): solo entonces se sabe que j es más potente. Es
    concordante si la predicción de j es mayor; un empate de predicciones cuenta
    medio. Con todas las mediciones exactas es el índice de concordancia
    ordinario. Devuelve `nan` si no hay pares comparables.
    """
    inf = np.asarray(limite_inferior, dtype=float)
    sup = np.asarray(limite_superior, dtype=float)
    pred = np.asarray(prediccion, dtype=float)
    concordantes = 0.0
    comparables = 0
    for a in range(0, len(pred), bloque):
        s = sup[a:a + bloque, None]
        p = pred[a:a + bloque, None]
        comparable = s < inf[None, :]          # i (fila) es menos potente que j
        comparables += int(comparable.sum())
        concordantes += float((comparable & (p < pred[None, :])).sum())
        concordantes += 0.5 * float((comparable & (p == pred[None, :])).sum())
    return concordantes / comparables if comparables else float("nan")


def spearman_exactas(limite_inferior: np.ndarray, limite_superior: np.ndarray,
                     prediccion: np.ndarray) -> tuple[float, int]:
    """Spearman sobre las mediciones exactas; devuelve `(rho, n)`."""
    e = _exactas(limite_inferior, limite_superior)
    pred = np.asarray(prediccion, dtype=float)[e]
    # Con una predicción constante (un modelo que siempre devuelve el promedio) el
    # orden no está definido: se declara nan en vez de dejar que scipy avise cada vez.
    if e.sum() < 3 or np.ptp(pred) == 0:
        return float("nan"), int(e.sum())
    rho = spearmanr(np.asarray(limite_inferior, dtype=float)[e], pred).statistic
    return float(rho), int(e.sum())


def rmse_exactas(limite_inferior: np.ndarray, limite_superior: np.ndarray,
                 prediccion: np.ndarray) -> tuple[float, int]:
    """RMSE sobre las mediciones exactas; devuelve `(rmse, n)`. Es solo del
    subconjunto no censurado, y así hay que declararlo donde se reporte."""
    e = _exactas(limite_inferior, limite_superior)
    if not e.any():
        return float("nan"), 0
    error = (np.asarray(limite_inferior, dtype=float)[e]
             - np.asarray(prediccion, dtype=float)[e])
    return float(np.sqrt(np.mean(error ** 2))), int(e.sum())


def metricas_regresion(limite_inferior: np.ndarray, limite_superior: np.ndarray,
                       prediccion: np.ndarray) -> dict[str, float]:
    """Las tres métricas con las cuentas que permiten leerlas."""
    rho, n_rho = spearman_exactas(limite_inferior, limite_superior, prediccion)
    rmse, n_rmse = rmse_exactas(limite_inferior, limite_superior, prediccion)
    return {
        "concordancia": concordancia_harrell(limite_inferior, limite_superior, prediccion),
        "spearman_exactas": rho,
        "rmse_exactas": rmse,
        "n_exactas": n_rmse,
        "n_total": int(len(prediccion)),
        "fraccion_censurada": 1.0 - n_rmse / max(len(prediccion), 1),
    }

#### El Tobit

Supone que la potencia es normal alrededor de una combinación lineal de las entradas, $\text{pActividad} \sim \mathcal{N}(Xw + b,\ \sigma^2)$, y maximiza la verosimilitud de lo que cada fila **sí** dice: una medición exacta aporta la densidad en su valor; un `> 10 µM`, que en pActividad es $(-\infty, 5]$, aporta la probabilidad $\Phi\big((5-\mu)/\sigma\big)$ de caer por debajo de 5; un `< L`, la de caer por encima. Cada fila es un intervalo `[inferior, superior]`: exacta si los dos extremos coinciden, censurada si uno es infinito. `verosimilitud` devuelve menos la log-verosimilitud media más la penalización L2, y su gradiente analítico, para que L-BFGS optimice; las entradas se estandarizan dentro de `fit`, así que cada pliegue aprende su propia media y varianza.

In [3]:
_LOG_2PI = np.log(2.0 * np.pi)


def _log_phi(z):
    return -0.5 * z * z - 0.5 * _LOG_2PI


def _log_masa(zi, zs):
    """log(Φ(zs) − Φ(zi)) sin perder la cola: la diferencia se hace en la cola
    donde las dos probabilidades son pequeñas, no entre dos números cercanos a 1."""
    out = np.empty_like(zi)
    baja = zs <= 0
    alta = zi >= 0
    medio = ~(baja | alta)
    a, b = log_ndtr(zs[baja]), log_ndtr(zi[baja])
    out[baja] = a + np.log1p(-np.exp(b - a))
    a, b = log_ndtr(-zi[alta]), log_ndtr(-zs[alta])
    out[alta] = a + np.log1p(-np.exp(b - a))
    out[medio] = np.log(np.maximum(
        np.exp(log_ndtr(zs[medio])) - np.exp(log_ndtr(zi[medio])), 1e-300))
    return out


def verosimilitud(theta, X, inferior, superior, alfa):
    """Menos la log-verosimilitud media más la penalización, y su gradiente.

    `theta` = (pesos, sesgo, log σ).
    """
    n, p = X.shape
    w, b, s = theta[:p], theta[p], theta[p + 1]
    sigma = np.exp(s)
    mu = X @ w + b

    exacta = inferior == superior
    izq = np.isneginf(inferior) & np.isfinite(superior)       # (−∞, U]
    der = np.isfinite(inferior) & np.isposinf(superior)       # [V, +∞)
    otra = ~(exacta | izq | der) & ~(np.isneginf(inferior) & np.isposinf(superior))

    ll = np.zeros(n)
    d_mu = np.zeros(n)
    d_sigma = np.zeros(n)

    z = (superior[exacta] - mu[exacta]) / sigma
    ll[exacta] = -s + _log_phi(z)
    d_mu[exacta] = z / sigma
    d_sigma[exacta] = (z * z - 1.0) / sigma

    # (−∞, U]: log Φ(z_U); la razón φ/Φ en escala logarítmica por estabilidad
    z = (superior[izq] - mu[izq]) / sigma
    ll[izq] = log_ndtr(z)
    razon = np.exp(_log_phi(z) - ll[izq])
    d_mu[izq] = -razon / sigma
    d_sigma[izq] = -razon * z / sigma

    # [V, +∞): log Φ(−z_V)
    z = (inferior[der] - mu[der]) / sigma
    ll[der] = log_ndtr(-z)
    razon = np.exp(_log_phi(z) - ll[der])
    d_mu[der] = razon / sigma
    d_sigma[der] = razon * z / sigma

    if otra.any():                                  # intervalo finito, poco común
        zi = (inferior[otra] - mu[otra]) / sigma
        zs = (superior[otra] - mu[otra]) / sigma
        lm = _log_masa(zi, zs)
        ll[otra] = lm
        fi = np.exp(_log_phi(zi) - lm)
        fs = np.exp(_log_phi(zs) - lm)
        d_mu[otra] = (fi - fs) / sigma
        d_sigma[otra] = (fi * zi - fs * zs) / sigma

    perdida = -ll.mean() + 0.5 * alfa * float(w @ w)
    g_w = -(X.T @ d_mu) / n + alfa * w
    g_b = -d_mu.mean()
    g_s = -(d_sigma * sigma).mean()
    return perdida, np.concatenate([g_w, [g_b, g_s]])


class RegresionTobit:
    """Regresión lineal con censura por intervalos.

    No hereda de `BaseEstimator`: `fit` recibe los dos extremos del intervalo en
    lugar de un solo `y`, y una canalización de sklearn solo pasa uno.
    """

    def __init__(self, alfa: float = 1e-3, max_iter: int = 300):
        self.alfa = alfa
        self.max_iter = max_iter

    def fit(self, X, inferior, superior):
        X = np.asarray(X, dtype=float)
        inferior = np.asarray(inferior, dtype=float)
        superior = np.asarray(superior, dtype=float)
        # Un intervalo vacío (inferior > superior) es una evidencia incompatible,
        # no una medición: no tiene verosimilitud y se descarta.
        valida = inferior <= superior
        X, inferior, superior = X[valida], inferior[valida], superior[valida]
        self.n_descartadas_ = int((~valida).sum())
        self.media_ = X.mean(axis=0)
        self.escala_ = X.std(axis=0)
        self.escala_[self.escala_ == 0] = 1.0
        Z = (X - self.media_) / self.escala_

        exactas = inferior == superior
        y0 = float(np.mean(inferior[exactas])) if exactas.any() else 0.0
        s0 = float(np.log(np.std(inferior[exactas]))) if exactas.sum() > 1 else 0.0
        theta0 = np.concatenate([np.zeros(Z.shape[1]), [y0, s0]])
        r = minimize(verosimilitud, theta0, args=(Z, inferior, superior, self.alfa),
                     jac=True, method="L-BFGS-B", options={"maxiter": self.max_iter})
        p = Z.shape[1]
        self.pesos_, self.sesgo_, self.sigma_ = r.x[:p], r.x[p], float(np.exp(r.x[p + 1]))
        self.convergio_ = bool(r.success)
        return self

    def predict(self, X):
        """La potencia esperada (pActividad)."""
        Z = (np.asarray(X, dtype=float) - self.media_) / self.escala_
        return Z @ self.pesos_ + self.sesgo_

#### Los baselines y la validación

`SoloExactas` entrena un modelo de sklearn con las exactas y descarta las cotas; `ImputadaEnCota` toma cada cota como si fuera el valor medido (un `(−∞, 5]` entra como 5); `TobitComparable` es el Tobit con la penalización de Ridge escrita en su escala (`alfa = A / n`). `evaluar_baselines` hace la validación cruzada agrupada y devuelve una fila por (modelo, semilla, pliegue, estrato); `tabla` promedia.

In [4]:
class SoloExactas(BaseEstimator):
    """Entrena un modelo de sklearn con las mediciones exactas y descarta las cotas."""

    def __init__(self, modelo):
        self.modelo = modelo

    @property
    def necesita_grupos(self) -> bool:
        return getattr(self.modelo, "necesita_grupos", False)

    def fit(self, X, inferior, superior, grupos=None):
        exacta = np.isfinite(inferior) & (inferior == superior)
        X_ex = X[exacta]
        y_ex = inferior[exacta]
        if getattr(self.modelo, "necesita_grupos", False):
            g_ex = np.asarray(grupos)[exacta] if grupos is not None else None
            self.modelo.fit(X_ex, y_ex, grupos=g_ex)
        else:
            self.modelo.fit(X_ex, y_ex)
        return self

    def predict(self, X):
        return self.modelo.predict(X)


class ImputadaEnCota(BaseEstimator):
    """Entrena un modelo de sklearn con las exactas y con cada censurada **tomando su cota
    como si fuera el valor medido**: un `(−∞, 5]` entra como 5 y un `[6, ∞)` como 6. Es lo
    que haría quien no quiere tirar los inactivos y no implementa una verosimilitud; sirve
    de control del Tobit. Descarta las filas sin ninguna cota y las de cotas cruzadas."""

    def __init__(self, modelo):
        self.modelo = modelo

    @property
    def necesita_grupos(self) -> bool:
        return getattr(self.modelo, "necesita_grupos", False)

    def fit(self, X, inferior, superior, grupos=None):
        util = intervalo_util(inferior, superior)
        valor = np.where(np.isfinite(inferior), inferior, superior)
        X_ut = X[util]
        y_ut = valor[util]
        if getattr(self.modelo, "necesita_grupos", False):
            g_ut = np.asarray(grupos)[util] if grupos is not None else None
            self.modelo.fit(X_ut, y_ut, grupos=g_ut)
        else:
            self.modelo.fit(X_ut, y_ut)
        return self

    def predict(self, X):
        return self.modelo.predict(X)


class TobitComparable(RegresionTobit):
    """El Tobit con la penalización de Ridge escrita en su escala (`alfa = A / n`)."""

    def __init__(self, alfa_ridge: float = ALFA_RIDGE, max_iter: int = 300):
        super().__init__(alfa=0.0, max_iter=max_iter)
        self.alfa_ridge = alfa_ridge

    def fit(self, X, inferior, superior):
        valida = np.asarray(inferior) <= np.asarray(superior)
        self.alfa = self.alfa_ridge / max(int(valida.sum()), 1)
        return super().fit(X, inferior, superior)


def baselines() -> dict:
    return {
        "media": SoloExactas(DummyRegressor()),
        "ridge_exactas": SoloExactas(make_pipeline(StandardScaler(), Ridge(alpha=ALFA_RIDGE))),
        "ridge_cota_imputada": ImputadaEnCota(
            make_pipeline(StandardScaler(), Ridge(alpha=ALFA_RIDGE))),
        "tobit": TobitComparable(),
    }


def intervalo_util(inferior: np.ndarray, superior: np.ndarray) -> np.ndarray:
    """Las filas que informan algo: con alguna cota y sin cotas que se contradigan."""
    sin_cota = np.isneginf(inferior) & np.isposinf(superior)
    return ~sin_cota & (inferior <= superior)


In [5]:
def evaluar_baselines(X, inferior, superior, grupos, estratos, clase,
                      semillas: Sequence[int] = (0, 1, 2), pliegues: int = 5,
                      modelos: dict | None = None) -> pd.DataFrame:
    """Validación cruzada agrupada de los baselines (o de los modelos indicados);
    una fila por (modelo, semilla, pliegue, estrato).

    `clase` es la etiqueta derivada a 10 µM (1, 0 o NaN si el intervalo no decide); con ella
    se calculan MCC y AUROC cortando la potencia predicha en 5. Las métricas de regresión
    usan el intervalo completo: la concordancia con las censuradas, y Spearman y RMSE solo
    con las exactas. El estrato se reporta si tiene `MIN_POR_CLASE` casos de cada clase.
    """
    X = np.asarray(X, dtype=float)
    inferior = np.asarray(inferior, dtype=float)
    superior = np.asarray(superior, dtype=float)
    grupos = np.asarray(grupos)
    clase = np.asarray(clase, dtype=float)
    estratificar = np.where(np.isnan(clase), -1, clase).astype(int)
    filas = []
    for semilla in semillas:
        cv = StratifiedGroupKFold(pliegues, shuffle=True, random_state=semilla)
        for k, (ajuste, val) in enumerate(cv.split(X, estratificar, grupos), 1):
            catalogo = baselines() if modelos is None else {n: clone(m) for n, m in modelos.items()}
            for nombre, modelo in catalogo.items():
                reloj = time.perf_counter()
                if getattr(modelo, "necesita_grupos", False):
                    modelo.fit(X[ajuste], inferior[ajuste], superior[ajuste], grupos=grupos[ajuste])
                else:
                    modelo.fit(X[ajuste], inferior[ajuste], superior[ajuste])
                seg = time.perf_counter() - reloj
                pred = modelo.predict(X[val])
                rmse_ent, _ = rmse_exactas(inferior[ajuste], superior[ajuste],
                                           modelo.predict(X[ajuste]))
                grupos_val = [("todos", np.ones(len(val), dtype=bool))]
                grupos_val += [(e, np.asarray(estratos)[val] == e)
                               for e in np.unique(np.asarray(estratos)[val])]
                for estrato, sel in grupos_val:
                    c = clase[val][sel]
                    c = c[~np.isnan(c)]
                    if estrato != "todos" and min(c.sum(), (1 - c).sum()) < MIN_POR_CLASE:
                        continue
                    m = metricas_regresion(inferior[val][sel], superior[val][sel], pred[sel])
                    decide = ~np.isnan(clase[val][sel])
                    ok = len(np.unique(clase[val][sel][decide])) == 2
                    fila = {"modelo": nombre, "semilla": semilla, "pliegue": k,
                            "estrato": estrato, "seg": seg, "rmse_entrenamiento": rmse_ent,
                            "concordancia": m["concordancia"],
                            "spearman": m["spearman_exactas"], "rmse": m["rmse_exactas"],
                            "n_exactas": m["n_exactas"],
                            "mcc": np.nan, "auroc": np.nan}
                    if ok:
                        y = clase[val][sel][decide].astype(int)
                        p = pred[sel][decide]
                        fila["mcc"] = matthews_corrcoef(y, p >= CORTE_10_UM)
                        fila["auroc"] = roc_auc_score(y, p)
                    filas.append(fila)
    return pd.DataFrame(filas)


def tabla(res: pd.DataFrame, estrato: str = "todos") -> pd.DataFrame:
    """Media por modelo, ordenada por concordancia. `desv_*` es dispersión entre
    evaluaciones, no error estándar: las semillas comparten filas."""
    r = res[res["estrato"] == estrato]
    t = r.groupby("modelo")[["concordancia", "spearman", "rmse", "mcc", "auroc",
                             "rmse_entrenamiento", "seg"]].mean()
    t.insert(1, "desv_concordancia", r.groupby("modelo")["concordancia"].std())
    return t.sort_values("concordancia", ascending=False).round(3)


### Los datos

La unidad es el par compuesto-diana, con los mismos pares de entrenamiento y validación que usa el [04](04_Models.ipynb) (la prueba reservada no se toca) y la diana como columnas de 0 y 1. Lo que cambia respecto de la clasificación es qué filas entran: **ninguna se descarta por no decidir a 10 µM**, porque una cota como `> 1 µM` no decide a 10 µM pero sí informa a un modelo con censura. Salen solo las filas que no informan nada: las que no tienen ninguna cota y las de cotas que se contradicen (intervalo vacío).

In [6]:
pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d = pares[pares["particion"].ne("prueba")].merge(desc, on="smiles").reset_index(drop=True)
cols = [c for c in desc.columns if c != "smiles"]
dianas = pd.get_dummies(d["target_gene"])

inferior = d["limite_inferior"].to_numpy(float)
superior = d["limite_superior"].to_numpy(float)
util = intervalo_util(inferior, superior)
print(f"{len(d):,} pares de entrenamiento y validación · {(~util).sum():,} sin información "
      f"(sin cota o con cotas cruzadas) y salen")

d, dianas = d[util].reset_index(drop=True), dianas[util].reset_index(drop=True)
inferior, superior = inferior[util], superior[util]
X = np.hstack([d[cols].to_numpy("float32"), dianas.to_numpy("float32")])
nombres = cols + list(dianas.columns)
exacta = inferior == superior
clase = d["activo_10um"].to_numpy(float)          # la clase derivada a 10 µM; NaN si no decide
grupos = d["grupo_cv"].to_numpy()
procedencia = d["estrato"].to_numpy()             # solo_chembl, solo_npass, mixto
print(f"{len(d):,} pares · {exacta.sum():,} con potencia exacta ({exacta.mean():.1%}) · "
      f"{(~exacta).sum():,} solo con cotas · {np.isnan(clase).sum():,} no deciden a 10 µM")

56,411 pares de entrenamiento y validación · 170 sin información (sin cota o con cotas cruzadas) y salen


56,241 pares · 46,100 con potencia exacta (82.0%) · 10,141 solo con cotas · 1,880 no deciden a 10 µM


## 2. Cuatro baselines

| Baseline | Qué ve | Para qué |
|:--|:--|:--|
| `media` | las exactas; predice siempre su promedio | el piso: un modelo que no le gane no aprendió nada |
| `ridge_exactas` | solo las exactas | la regresión lineal que un baseline sencillo haría |
| `ridge_cota_imputada` | las exactas y las censuradas, con la cota tomada como el valor | lo ingenuo de quien no quiere tirar los inactivos ni implementar una verosimilitud |
| `tobit` | las exactas y las censuradas como cotas, con su verosimilitud | la misma recta, tratando la censura como lo que es |

Los tres últimos son la misma familia con la misma penalización (`alfa = 100` de Ridge, escrita en la escala de cada uno). **Se necesitan los dos Ridge para no confundir dos cosas:** el Tobit ve más filas que `ridge_exactas` *y* usa otra pérdida, así que su ventaja admite dos lecturas. `ridge_exactas` contra `ridge_cota_imputada` mide lo que vale no tirar los censurados; `ridge_cota_imputada` contra `tobit` mide lo que vale la verosimilitud. El Tobit supone que la potencia es normal alrededor de la recta y maximiza la verosimilitud de lo que cada fila sí dice: una exacta aporta su valor y un `> 10 µM` la probabilidad de caer por debajo de pActividad 5 ([`src/models/tobit.py`](../src/models/tobit.py)).

La validación es la del Avance 3: cinco pliegues agrupados por `grupo_cv` (andamio y esqueleto de conectividad), semillas 0 a 4 (cinco; en el Avance 3 eran tres, y aquí se agregan porque el estrato de solo NPASS, el que más se va a citar, es el que menos mediciones exactas tiene: 982 de sus 2,114 pares). Las métricas de regresión usan el intervalo completo; MCC y AUROC se calculan sobre la clase derivada, cortando la potencia predicha en 5 (= 10 µM), para poder compararlas con el baseline de clasificación. Un estrato se reporta solo si tiene 20 casos de cada clase.

In [7]:
SEMILLAS = (0, 1, 2, 3, 4)
res = evaluar_baselines(X, inferior, superior, grupos, procedencia, clase, semillas=SEMILLAS)
print(f"{len(res):,} filas: {res['modelo'].nunique()} modelos × {len(SEMILLAS)} semillas × 5 pliegues, por estrato")

400 filas: 4 modelos × 5 semillas × 5 pliegues, por estrato


## 3. Resultados

In [8]:
tabla_todos = tabla(res)
tabla_todos

,concordancia,desv_concordancia,spearman,rmse,mcc,auroc,rmse_entrenamiento,seg
modelo,,,,,,,,
tobit,0.700,0.007,0.512,1.234,0.363,0.792,1.181,7.693
ridge_cota_imputada,0.698,0.007,0.522,1.186,0.293,0.788,1.139,1.356
ridge_exactas,0.693,0.007,0.531,1.156,0.225,0.776,1.115,1.018
media,0.500,0.000,NaN,1.338,0.000,0.500,1.338,0.399


### Por procedencia

Un promedio puede esconder cómo le va a los naturales: el control del Avance 3 mostró que el modelo separa mucho peor los pares que solo tiene NPASS.

In [9]:
por_procedencia = {e: tabla(res, e) for e in ("solo_chembl", "mixto", "solo_npass")
                   if (res["estrato"] == e).any()}
pd.concat(por_procedencia, names=["procedencia"])[
    ["concordancia", "desv_concordancia", "spearman", "rmse", "mcc", "auroc"]]

concordancia  desv_concordancia  spearman  \
procedencia modelo                                                           
solo_chembl tobit                       0.688              0.009     0.489   
            ridge_cota_imputada         0.687              0.008     0.497   
            ridge_exactas               0.681              0.007     0.504   
            media                       0.500              0.000       NaN   
mixto       tobit                       0.530              0.034     0.015   
            ridge_cota_imputada         0.525              0.033     0.024   
            ridge_exactas               0.510              0.029     0.043   
            media                       0.500              0.000       NaN   
solo_npass  tobit                       0.639              0.036     0.189   
            ridge_cota_imputada         0.586              0.032     0.221   
            ridge_exactas               0.533              0.030     0.237   
            media                       0.500              0.000       NaN   

                                  rmse    mcc  auroc  
procedencia modelo                                    
solo_chembl tobit                1.215  0.328  0.780  
            ridge_cota_imputada  1.171  0.249  0.776  
            ridge_exactas        1.148  0.173  0.762  
            media                1.325  0.000  0.500  
mixto       tobit                1.186  0.021  0.509  
            ridge_cota_imputada  1.085  0.007  0.506  
            ridge_exactas        1.092 -0.007  0.499  
            media                1.498  0.000  0.500  
solo_npass  tobit                1.585  0.115  0.624  
            ridge_cota_imputada  1.473  0.070  0.576  
            ridge_exactas        1.350  0.013  0.527  
            media                1.638  0.000  0.500

### ¿Sobreajusta?

El error en las exactas de entrenamiento contra el de validación. Una brecha grande diría que el modelo memoriza; un modelo lineal con penalización no debería.

In [10]:
t = tabla_todos[["rmse_entrenamiento", "rmse"]].rename(columns={"rmse": "rmse_validacion"})
t["brecha"] = (t["rmse_validacion"] - t["rmse_entrenamiento"]).round(3)
t

,rmse_entrenamiento,rmse_validacion,brecha
modelo,,,
tobit,1.181,1.234,0.053
ridge_cota_imputada,1.139,1.186,0.047
ridge_exactas,1.115,1.156,0.041
media,1.338,1.338,0.000


### Qué características pesan

Los coeficientes de la recta, sobre entradas estandarizadas, ajustada con todo entrenamiento y validación. Si la censura cambiara qué importa, los coeficientes diferirían; se comparan las quince entradas de mayor peso absoluto del Tobit con las de cada Ridge.

In [11]:
ridge = baselines()["ridge_exactas"].fit(X, inferior, superior)
imputada = baselines()["ridge_cota_imputada"].fit(X, inferior, superior)
tobit = baselines()["tobit"].fit(X, inferior, superior)
coef = pd.DataFrame({"ridge": ridge.modelo[-1].coef_, "imputada": imputada.modelo[-1].coef_,
                     "tobit": tobit.pesos_}, index=nombres)

top = {k: set(coef[k].abs().nlargest(15).index) for k in coef}
for otro in ("ridge", "imputada"):
    print(f"Tobit contra {otro}: {len(top['tobit'] & top[otro])} de las 15 entradas de mayor peso "
          f"coinciden; correlación de los coeficientes {coef['tobit'].corr(coef[otro]):.3f}")
print(f"σ del Tobit (ruido que atribuye a la medición): {tobit.sigma_:.2f} unidades de pActividad")
coef.reindex(coef["tobit"].abs().sort_values(ascending=False).index).head(15).round(3)

Tobit contra ridge: 9 de las 15 entradas de mayor peso coinciden; correlación de los coeficientes 0.900
Tobit contra imputada: 10 de las 15 entradas de mayor peso coinciden; correlación de los coeficientes 0.957
σ del Tobit (ruido que atribuye a la medición): 1.31 unidades de pActividad


,ridge,imputada,tobit
BertzCT,-0.360,-0.394,-0.449
NumHAcceptors,0.287,0.373,0.429
HallKierAlpha,-0.338,-0.438,-0.403
Chi3v,0.308,0.403,0.385
SMR_VSA3,-0.308,-0.336,-0.366
Chi2n,-0.342,-0.355,-0.339
SlogP_VSA6,-0.390,-0.393,-0.337
MAOA,-0.172,-0.216,-0.294
fr_NH1,0.374,0.308,0.289
LRRK2,0.183,0.211,0.254


### El piso y el techo

El **piso** es `media`. El **techo** es el ruido propio de la medición: repetir el mismo ensayo da diferencias de unas 0.7 unidades logarítmicas en IC50 y 0.5 en Ki (`kalliokoski2013ic50`, `kramer2012ki`; con curación mínima el ruido es mayor, `landrum2024ruido`). Un RMSE muy por debajo de eso no sería un buen resultado sino una señal de fuga.

In [12]:
print(f"techo de reproducibilidad: {RUIDO_EXPERIMENTAL_IC50} (IC50) y {RUIDO_EXPERIMENTAL_KI} (Ki) unidades logarítmicas")
tabla_todos[["rmse"]].assign(por_encima_del_techo_ic50=lambda t: (t["rmse"] > RUIDO_EXPERIMENTAL_IC50))

techo de reproducibilidad: 0.7 (IC50) y 0.5 (Ki) unidades logarítmicas


,rmse,por_encima_del_techo_ic50
modelo,,
tobit,1.234,True
ridge_cota_imputada,1.186,True
ridge_exactas,1.156,True
media,1.338,True


## 4. Qué queda establecido, y qué no

**Lo que establece.**

1. **Hay señal más allá del promedio, y es modesta.** Las tres rectas ordenan mejor que el azar: concordancia de 0.693 a 0.700 contra 0.500 de `media`, y AUROC de la clase derivada de 0.776 a 0.792. Pero el RMSE sobre las exactas es de 1.16 a 1.23 unidades de pActividad contra 1.34 del promedio: el error es de 1.7 a 1.8 veces el ruido propio de la medición (0.7), lejos del techo y sin señal de fuga.
2. **En el agregado las tres rectas no se distinguen por concordancia** (0.693, 0.698 y 0.700, con una dispersión de 0.007 entre evaluaciones). Sí se ordenan por el MCC de la clase derivada (0.225, 0.293 y 0.363 de `ridge_exactas`, `ridge_cota_imputada` y `tobit`) y por el AUROC (0.776, 0.788 y 0.792).
3. **En los pares que solo tiene NPASS la concordancia sube en dos escalones iguales:** 0.533 con `ridge_exactas`, 0.586 con la cota imputada y 0.639 con el Tobit (dispersión de 0.030 a 0.036). Cada escalón vale 0.053, de 1.5 a 1.8 veces la dispersión; el total, 0.106, unas 3.4 veces. Esto contesta la pregunta que el diseño anterior no separaba: **la ganancia sobre los naturales tiene dos partes de tamaño parecido.** Una llega solo por no tirar los censurados (la cota imputada, que no implementa ninguna verosimilitud); la otra, por tratar la censura con su verosimilitud. Ninguno de los dos escalones, tomado solo, supera con claridad el ruido de partición; juntos sí.
4. **La censura cuesta exactitud en las exactas, y el costo crece con el tratamiento:** RMSE de 1.156, 1.186 y 1.234 y Spearman de 0.531, 0.522 y 0.512, en el mismo orden.
5. **Donde no hay naturales no hay diferencia:** en los pares de solo ChEMBL la concordancia es 0.681, 0.687 y 0.688, y en los mixtos ninguna recta supera 0.53.
6. **No sobreajustan.** La brecha entre el RMSE de entrenamiento y el de validación es de 0.04 a 0.05.
7. **Las rectas coinciden en lo que pesa, y el Tobit se parece más a la cota imputada** (correlación de coeficientes 0.957, diez de las quince entradas de mayor peso) **que a la recta con exactas** (0.900, nueve de quince). El σ que el Tobit atribuye a la medición, 1.31, es mayor que el techo de 0.7: dice cuánto deja sin explicar la recta, no que la medición sea tan ruidosa.
8. **El Tobit tarda unas ocho veces más que Ridge** (5.8 s contra 0.7 s por ajuste). Esos tiempos se midieron con otra ejecución corriendo en la misma máquina: sirven para el orden, no como cifra.

**El desempeño mínimo para lo que sigue.** No hay un histórico contra el cual comparar. Los modelos del 04b tendrán que superar el mejor de cada métrica entre estos baselines: concordancia 0.700, Spearman 0.531 y RMSE 1.156 sobre las exactas.

**Lo que no establece.**

- **No es el modelo final.** La comparación de familias es del 04b.
- **No dice que la verosimilitud valga más que no tirar los censurados**, ni al revés: los dos escalones son del mismo tamaño y cada uno cabe en la dispersión. Lo que sí dice es que los naturales mejoran cuando los inactivos censurados se usan, de cualquier manera.
- **El RMSE y el Spearman miran solo las exactas**, así que no dicen nada de qué tan bien se predicen las censuradas; para eso está la concordancia.
- **El MCC y el AUROC no son comparables con los del baseline de clasificación del [03](03_Baseline.ipynb):** son otro modelo, y aquí se calculan sobre los pares que deciden a 10 µM, con una potencia predicha cortada en 5.
- **Cinco semillas que comparten filas:** la dispersión que se reporta no es un error estándar, y el estrato de solo NPASS tiene 2,114 pares, de los cuales solo 982 tienen una medición exacta.


In [13]:
# Registro en experimentos/ (decisión 2.9: la potencia como variable principal).
from src.tracking import Corrida

with Corrida("baseline-regresion", decision="2.9",
             params={"modelos": list(baselines()), "pliegues": 5, "alfa_ridge": ALFA_RIDGE,
                     "objetivo": "pActividad, con las censuradas como cotas (Tobit)",
                     "corte_mcc": 5.0, "estratos": "procedencia"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "fase2_pares.parquet",
                    PROCESADOS / "fase2_descriptores_limpios.parquet"]) as c:
    c.metricas({"todos": tabla_todos.to_dict(orient="index"),
                **{e: t.to_dict(orient="index") for e, t in por_procedencia.items()}})

print(c.carpeta.name)

2026-10-25-ingrid-2.9-baseline-regresion


# Declaración de uso de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Escribir el módulo del baseline (`src/models/baseline_regresion.py`), el modelo de Tobit (`src/models/tobit.py`), las métricas de regresión (`src/evaluation/regresion.py`), sus pruebas y este cuaderno | Todas | Pruebas automáticas del gradiente, del signo de la censura y de los baselines (`tests/test_tobit.py`, `tests/test_regresion.py`, `tests/test_baseline_regresion.py`). **Lo que el equipo cambió:** el diseño original comparaba la recta con exactas contra el Tobit y mezclaba dos efectos; la revisión de Artemio Padilla pidió el control de la cota imputada y las cinco semillas, y se agregaron. La lectura de la sección 4 y la ejecución del cuaderno las revisa Ingrid Pamela Ruiz Puga (por confirmar en la revisión del PR) |

### Contribución individual

| Integrante | Contribución a este cuaderno |
|---|---|
| Ingrid Pamela Ruiz Puga | Planteó el baseline de regresión con y sin censura y lo ejecutó; con apoyo de Claude Code escribió el modelo de Tobit, los cuatro baselines, las métricas y la validación; atendió la crítica de Artemio Padilla, que añadió el control de la cota imputada y las cinco semillas y cambió la lectura de la sección 4 |

Las salidas de la herramienta se usaron como insumo de trabajo y el equipo asume la responsabilidad por la exactitud, la integridad y el contenido final del cuaderno.
